<a href="https://colab.research.google.com/github/kjahan/nlp_tips_and_trick/blob/main/notebooks/explore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time

## For loop in python

https://stackoverflow.com/questions/8097408/why-python-is-so-slow-for-a-simple-for-loop

In [3]:
start = time.time()

lst = []

for i in range(100000):
  x = i**2
  lst.append(x)

print("Runtime: {}".format((time.time() - start)))

Runtime: 0.04195070266723633


In [4]:
start = time.time()

lst = [i**2 for i in range(100000)]

print("Runtime: {}".format((time.time() - start)))

Runtime: 0.030115127563476562


In [2]:
start = time.time()

total = 0.0
for i in range(9999): # xrange is slower according
    for j in range(1, 9999):            #to my test but more memory-friendly.
        total += (i / j)

print(total)
print("Runtime: {}".format((time.time() - start)))

489223499.9991652
Runtime: 14.51483678817749


In [5]:
%%capture
!pip install cython

In [6]:
%load_ext Cython

In [7]:
%%cython
def calculate():
  total = 0.0
  for i in range(9999): # xrange is slower according
      for j in range(1, 9999):            #to my test but more memory-friendly.
          total += (i / j)
  return total
# def calculate_squares():
#     lst = []

#     for i in range(100000):
#         lst.append(i**2)

#     return lst


In [9]:
%%cython
cpdef calculate_total():
    cdef double total = 0.0
    cdef int i, j

    for i in range(9999):
        for j in range(1, 9999):
            total += (i / j)

    return total

In [8]:
start = time.time()
result = calculate()
print(result)
print("Runtime: {}".format((time.time() - start)))

489223499.9991652
Runtime: 4.600732088088989


In [10]:
start = time.time()
result = calculate_total()
print(result)
print("Runtime: {}".format((time.time() - start)))

489223499.9991652
Runtime: 0.12791848182678223


In [12]:
speedup = 14.51/.1279
speedup

113.44800625488662

## MSMARCO Models

https://www.sbert.net/docs/pretrained-models/msmarco-v3.html

In [14]:
!pip install -U sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.0 MB/s eta 0:00:00
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=bb73c61ac9902ef6ee00bd87f525dbd37f37e871cd173a8d9fb349cf5dd10809
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence-transformers


In [17]:
from sentence_transformers import SentenceTransformer, util

#  A common value for BERT & Co. are 512 word pieces, which correspond to about 300-400 words (for English).
# Longer texts than this are truncated to the first x word pieces.
# By default, the provided methods use a limit fo 128 word pieces, longer inputs will be truncated
# the runtime and the memory requirement grows quadratic with the input length - we'll have to play around with this

# Change the length to 200
model = SentenceTransformer("sentence-transformers/msmarco-distilbert-base-v3")
# it seems max seq len is 510
# https://huggingface.co/sentence-transformers/msmarco-distilbert-base-v3
model.max_seq_length = 510
print(model.max_seq_length)

510


## Quora dataset

https://www.kaggle.com/c/quora-question-pairs/data?select=test.csv.zip

In [31]:
corpus = []
s1, s2 = "Who first masturbation experience?","What is your initial masturbation experience?"
corpus.extend([s1, s2])

s1, s2 = "What is year benefit of advertisement?","Which one advertising?"
corpus.extend([s1, s2])

s1, s2 = "What are best kids best games?","What are the best types of educational games for children and developers do they teach?"
corpus.extend([s1, s2])

s1, s2 = "Which anime characters have t red eyes?","What characters have red hair?"
corpus.extend([s1, s2])

In [32]:
embeddings = model.encode(
    corpus, show_progress_bar=True, device="cuda", convert_to_numpy=False
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
len(embeddings)

4

In [27]:
from numpy import dot
from numpy.linalg import norm

In [29]:
a, b = embeddings[0].cpu(), embeddings[1].cpu()
cos_sim = dot(a, b)/(norm(a)*norm(b))
print("cos_sim: {}".format(cos_sim))


cos_sim: 0.8462244272232056


In [30]:
a, b = embeddings[2].cpu(), embeddings[3].cpu()
cos_sim = dot(a, b)/(norm(a)*norm(b))
print("cos_sim: {}".format(cos_sim))

cos_sim: 0.5051702260971069


In [33]:
a, b = embeddings[4].cpu(), embeddings[5].cpu()
cos_sim = dot(a, b)/(norm(a)*norm(b))
print("cos_sim: {}".format(cos_sim))

cos_sim: 0.5514034628868103


In [34]:
a, b = embeddings[6].cpu(), embeddings[7].cpu()
cos_sim = dot(a, b)/(norm(a)*norm(b))
print("cos_sim: {}".format(cos_sim))

cos_sim: 0.5767778158187866
